# Food Trust -- WHO NPPM Category Thresholds Build (Epic 3)

Builds `who_nppm_thresholds.db`, a SQLite database of the WHO Regional
Office for Europe's **Nutrient and Promotion Profile Model (NPPM), 2022**
composition thresholds for commercial complementary foods (6-36 months).

This is the table that actually drives Epic 3's US3.1 red/green
comparison cards, for the reason worked out earlier in this project: a
product's real nutrient values in `products.db` are stored **per 100 g of
product**, and NPPM's Table 1 thresholds are the only source in this
project that is also expressed on a **per-100 kcal-of-product** basis --
i.e. still describing the product itself, not a child's whole-day intake
-- so it is the one reference table that is actually comparable to a
product row. (`reference_values.db`, built earlier for Epic 3, stores
NHMRC/WHO Sugars figures as **per-day totals** or **%energy of a whole
day's diet** -- a different basis that cannot be reconciled with a
per-100g product value by unit conversion alone, since that would require
knowing how much of the product a child actually eats. See that
notebook's own note, and Epic 6 US6.2's acceptance criteria, which
describes exactly this "units/measurement bases do not match" case.)

Source: **WHO-EURO-2022-6681-46447-67287-eng.pdf**, Table 1 (pages 18-19
of the PDF), transcribed via `pdfplumber` (see the build cell's own
docstring for why -- `pdftotext -layout` scrambled this table's rotated
column headers).

## What this table covers, and what it deliberately does not

NPPM Table 1 gives thresholds for **5 of Epic 3's 8 nutrients only**:
Energy density, Sodium, Total sugar, Total protein, Total fat. It has
nothing for Saturated fat, Carbohydrate or Dietary fibre -- those three
stay backed by `reference_values.db` (which itself has no NHMRC value for
Saturated fat at any age -- a genuine, confirmed source gap, not a
build-time omission).

NPPM organises its thresholds by **13 product subcategories** (how the
product is composed/labelled), not by the child's age band the way
`reference_values.db` does. Two of the 13 -- **7 Confectionery** and
**8 Drinks** -- are marked by WHO as "not appropriate for promotion" to
this age group at all, so they carry no numeric thresholds; this table
still records them (with `eligible_for_promotion = 0`) so the app can
show an honest "this category shouldn't be marketed to this age" message
rather than silently having no row.

Run this notebook with its own folder (`reference_value/`) as the working
directory.

## Build who_nppm_thresholds.db

In [1]:
"""
Builds who_nppm_thresholds.db from WHO NPPM 2022's Table 1 (NPPM Part A:
content and front-of-pack labelling) -- the source that actually drives
Epic 3's US3.1 red/green comparison cards for the 5 nutrients it covers
(Energy density, Sodium, Total sugar, Total protein, Total fat).

Transcribed directly from a pdfplumber word-position extraction of
WHO-EURO-2022-6681-46447-67287-eng.pdf, pages 18-19 (the page/column
layout in pdftotext -layout mangled this table -- rotated vertical
headers overlapped the data columns -- pdfplumber gave clean, correctly
ordered row text instead, cross-checked against the 13 category rows and
the footnote text for internal consistency, e.g. footnote 12 confirms
which categories have a protein MAXIMUM vs a protein MINIMUM).

IMPORTANT -- this table is NOT directly per-100g-of-product like
products.db. NPPM expresses Sodium/Total sugar/Total protein/Total fat as
per-100kcal-of-product, not per-100g. To compare a product from
products.db against a threshold here, the app must first convert:
    nutrient_per_100kcal = nutrient_per_100g / energy_kcal_100g * 100
using that SAME product's own energy_kcal_100g. This conversion is a
runtime step for the backend, not something this table can precompute
(it depends on each individual product's own energy density).

IMPORTANT -- Energy density itself is NOT on one consistent basis across
categories, and this is a genuine, irreducible difference (not a units
bug to fix): most categories use kcal/100g of product "as eaten", but the
two snack categories (5.1, 5.2) use kcal PER SERVE instead -- and NPPM
does not define a fixed serve size in grams, so this cannot be converted
to kcal/100g without more information. Both units are kept, clearly
labelled, not force-unified.

Schema (long/tidy, one row per distinct threshold):
  category_code, category_name, category_description, age_label
  eligible_for_promotion (0/1 -- categories 7 Confectionery and 8 Drinks
    are marked "Not appropriate for promotion" in the source at all --
    stored as a single row each with eligible_for_promotion=0 and no
    numeric thresholds, rather than fabricating limits that do not exist)
  nutrient (energy_density / sodium / total_sugar_pct_energy /
    added_sugar / total_protein / total_fat / fruit_content_pct_weight)
  condition ('<=' / '>=' / 'not_permitted' / 'not_specified')
  value, unit
  value_note (every conditional exception transcribed in full: sodium's
    cheese-named exception, protein's dual g/100kcal + %-by-weight
    requirement for categories 4.3-4.5, category 1/5.2's protein CAP
    instead of minimum, energy density's per-serve vs per-100g split,
    the purée subtypes' narrower 6-12 month age window, etc.)
  source
"""
import sqlite3, tempfile, shutil, os
from pathlib import Path

BASE_DIR = Path.cwd()
OUT_DB = BASE_DIR / "who_nppm_thresholds.db"

SOURCE = "WHO 2022 NPPM (Nutrient and Promotion Profile Model), Table 1"

# One dict per category, transcribed from the clean pdfplumber extraction.
# cond values: '<=', '>=', 'not_permitted', 'not_specified'
# Each nutrient field is (condition, value, unit, note) or None if the
# source genuinely gives nothing for that nutrient/category.
CATEGORIES = [
    {
        "code": "1", "name": "Dry cereals and starches",
        "description": "Dry or powdered cereal/starch to be prepared with milk (or equivalent non-sweet liquid) or water, or eaten dry, e.g. instant porridge, muesli, baby rice, dry pasta. Excludes crackers/rusks (cat. 5.2).",
        "age_label": "6-36 months", "eligible": True,
        "energy_density": ("gte", 80, "kcal/100g", "as eaten"),
        "sodium": ("lte", 50, "mg/100kcal", None),
        "total_sugar_pct_energy": (None, None, "%energy", "No separate %energy sugar cap specified; added free sugar itself is banned outright (see added_sugar)."),
        "added_sugar": ("not_permitted", None, None, "No added free sugars or sweeteners permitted at all (WHO NPPM footnote 1 defines the full list of what counts as added sugar)."),
        "total_protein": ("lte", 5.5, "g/100kcal", "This is a CAP, not a minimum -- only applies if the product contains a high-protein food such as added milk (footnote 12i). No protein requirement otherwise for this category."),
        "total_fat": ("lte", 4.5, "g/100kcal", "as eaten; OR <=10% of dry weight if the product is intended to be prepared/eaten with milk."),
        "fruit_content_pct_weight": (None, None, "%weight", "Not applicable to this category."),
        "high_sugar_flag_note": "Flag required if sugar exceeds the 30% total-energy variant (=7.5g/100kcal, footnote 8).",
    },
    {
        "code": "2", "name": "Dairy foods, desserts and cereals",
        "description": "The largest ingredient is dairy and fruit content is <=5%, e.g. porridge, rice pudding, yogurt, fromage frais, custard. If fruit content >5%, use category 3.1 instead.",
        "age_label": "6-36 months", "eligible": True,
        "energy_density": ("gte", 60, "kcal/100g", None),
        "sodium": ("lte", 50, "mg/100kcal", "OR <=100mg/100kcal if cheese is named in the product name."),
        "total_sugar_pct_energy": (None, None, "%energy", "No separate %energy sugar cap specified; added free sugar itself is banned outright (see added_sugar)."),
        "added_sugar": ("not_permitted", None, None, "No added free sugars or sweeteners permitted at all."),
        "total_protein": (None, None, "g/100kcal", "No protein requirement for this category."),
        "total_fat": ("lte", 4.5, "g/100kcal", None),
        "fruit_content_pct_weight": ("lte", 5, "%weight", "Max 2% if dried fruit."),
        "high_sugar_flag_note": "Flag required if sugar exceeds the 40% total-energy variant (=10g/100kcal, footnote 8).",
    },
    {
        "code": "3.1", "name": "Fruit-containing product (purees/smoothies/fruit desserts)",
        "description": "Any product containing >5% fruit (except dry cereals, low-fruit dairy, or snacks), e.g. apple puree, fruit and yogurt, fruit custard, porridge with >5% fruit.",
        "age_label": "6-36 months (purees: 6-12 months only)", "eligible": True,
        "energy_density": ("gte", 60, "kcal/100g", None),
        "sodium": ("lte", 50, "mg/100kcal", None),
        "total_sugar_pct_energy": (None, None, "%energy", "No separate %energy sugar cap specified; added free sugar itself is banned outright (see added_sugar)."),
        "added_sugar": ("not_permitted", None, None, "No added free sugars or sweeteners permitted at all."),
        "total_protein": (None, None, "g/100kcal", "No protein requirement for this category."),
        "total_fat": ("lte", 4.5, "g/100kcal", None),
        "fruit_content_pct_weight": (None, None, "%weight", "No upper cap given -- this category is fruit-based by definition (>5% fruit is the category-defining condition itself)."),
        "high_sugar_flag_note": "Flag required if sugar exceeds the 30% total-energy variant (=7.5g/100kcal, footnote 8).",
    },
    {
        "code": "3.2", "name": "Vegetable-only product",
        "description": "Single or mixed vegetables or legumes, e.g. spinach & pea puree, mashed potato & carrot. Excludes foods containing added starch/fat/dairy (cat. 4.1).",
        "age_label": "6-36 months (purees: 6-12 months only)", "eligible": True,
        "energy_density": (None, None, "% added water", "<=25% added water (a different metric to the kcal/100g used elsewhere in this table -- not convertible to it)."),
        "sodium": ("lte", 50, "mg/100kcal", None),
        "total_sugar_pct_energy": (None, None, "%energy", "No separate %energy sugar cap specified; added free sugar itself is banned outright (see added_sugar)."),
        "added_sugar": ("not_permitted", None, None, "No added free sugars or sweeteners permitted at all."),
        "total_protein": (None, None, "g/100kcal", "No protein requirement for this category."),
        "total_fat": ("lte", 4.5, "g/100kcal", None),
        "fruit_content_pct_weight": ("not_permitted", None, "%weight", "No fruit permitted -- this is a vegetable-only category by definition."),
        "high_sugar_flag_note": "Flag required if sugar exceeds the 30% total-energy variant (=7.5g/100kcal, footnote 8).",
    },
    {
        "code": "4.1", "name": "Savoury meal WITHOUT protein or cheese named",
        "description": "Vegetables/legumes and/or cereals/starches. May contain a protein source, dairy or fats, e.g. vegetable rice, lasagne, pesto sauce for pasta.",
        "age_label": "6-36 months (purees: 6-12 months only)", "eligible": True,
        "energy_density": ("gte", 60, "kcal/100g", None),
        "sodium": ("lte", 50, "mg/100kcal", None),
        "total_sugar_pct_energy": ("lte", 15, "%energy", None),
        "added_sugar": ("not_permitted", None, None, "No added free sugars or sweeteners permitted at all."),
        "total_protein": ("gte", 3, "g/100kcal", "Minimum, footnotes 12iii/13."),
        "total_fat": ("lte", 4.5, "g/100kcal", "Max 2% of dry weight."),
        "fruit_content_pct_weight": ("lte", 5, "%weight", None),
        "high_sugar_flag_note": "Not applicable -- this category already has an explicit <=15% total-sugar-%energy cap, so the separate front-of-pack flag mechanism does not apply.",
    },
    {
        "code": "4.2", "name": "Savoury meal WITH CHEESE named, no other protein",
        "description": "Cheese and no other proteins are in the product name, e.g. cheese pasta, tomato & mozzarella pasta sauce.",
        "age_label": "6-36 months (purees: 6-12 months only)", "eligible": True,
        "energy_density": ("gte", 60, "kcal/100g", None),
        "sodium": ("lte", 100, "mg/100kcal", "Higher allowance than other savoury-meal subcategories because cheese is named."),
        "total_sugar_pct_energy": ("lte", 15, "%energy", None),
        "added_sugar": ("not_permitted", None, None, "No added free sugars or sweeteners permitted at all."),
        "total_protein": ("gte", 3, "g/100kcal", "Minimum, footnotes 12iii/13."),
        "total_fat": ("lte", 6, "g/100kcal", "Higher allowance than 4.1 because cheese is named. Max 2% of dry weight."),
        "fruit_content_pct_weight": ("lte", 5, "%weight", None),
        "high_sugar_flag_note": "Not applicable -- this category already has an explicit <=15% total-sugar-%energy cap.",
    },
    {
        "code": "4.3", "name": "Savoury meal, protein source NOT named first",
        "description": "Protein source is not the first named food, e.g. pea & lamb curry, tomato & beef sauce for pasta.",
        "age_label": "6-36 months (purees: 6-12 months only)", "eligible": True,
        "energy_density": ("gte", 60, "kcal/100g", None),
        "sodium": ("lte", 50, "mg/100kcal", "OR <=100mg/100kcal if cheese is named in the product name."),
        "total_sugar_pct_energy": ("lte", 15, "%energy", None),
        "added_sugar": ("not_permitted", None, None, "No added free sugars or sweeteners permitted at all."),
        "total_protein": ("gte", 3, "g/100kcal", "Dual requirement, footnote 12iv: also must be >=8% of total product weight from the named protein source."),
        "total_fat": ("lte", 4.5, "g/100kcal", "Max 2% of dry weight."),
        "fruit_content_pct_weight": ("lte", 5, "%weight", None),
        "high_sugar_flag_note": "Not applicable -- this category already has an explicit <=15% total-sugar-%energy cap.",
    },
    {
        "code": "4.4", "name": "Savoury meal, protein source named FIRST",
        "description": "e.g. Rabbit & potato, beef soup, tasty chicken risotto, chicken & cheese pasta, beef sauce for pasta.",
        "age_label": "6-36 months (purees: 6-12 months only)", "eligible": True,
        "energy_density": ("gte", 60, "kcal/100g", None),
        "sodium": ("lte", 50, "mg/100kcal", "OR <=100mg/100kcal if cheese is named in the product name."),
        "total_sugar_pct_energy": ("lte", 15, "%energy", None),
        "added_sugar": ("not_permitted", None, None, "No added free sugars or sweeteners permitted at all."),
        "total_protein": ("gte", 4, "g/100kcal", "Dual requirement, footnote 12iv: also must be >=10% of total product weight from the named protein source."),
        "total_fat": ("lte", 6, "g/100kcal", "Max 2% of dry weight."),
        "fruit_content_pct_weight": ("lte", 5, "%weight", None),
        "high_sugar_flag_note": "Not applicable -- this category already has an explicit <=15% total-sugar-%energy cap.",
    },
    {
        "code": "4.5", "name": "Protein source is ONLY named food",
        "description": "Pureed cooked meat. May contain a small quantity of grain/starch not in the product name, e.g. 'rabbit' or 'lamb' with some added rice flour or corn starch.",
        "age_label": "6-36 months (purees: 6-12 months only)", "eligible": True,
        "energy_density": ("gte", 60, "kcal/100g", None),
        "sodium": ("lte", 50, "mg/100kcal", None),
        "total_sugar_pct_energy": ("lte", 15, "%energy", None),
        "added_sugar": ("not_permitted", None, None, "No added free sugars or sweeteners permitted at all."),
        "total_protein": ("gte", 7, "g/100kcal", "Dual requirement, footnote 12iv: also must be >=40% of total product weight from the named protein source."),
        "total_fat": ("lte", 6, "g/100kcal", "Max 2% of dry weight."),
        "fruit_content_pct_weight": ("lte", 5, "%weight", None),
        "high_sugar_flag_note": "Not applicable -- this category already has an explicit <=15% total-sugar-%energy cap.",
    },
    {
        "code": "5.1", "name": "Fruit snacks",
        "description": "Fresh fruit or whole dry fruit pieces, e.g. plain dry apple slices or raisins. Excludes pulverised/pureed dry fruits (cat. 7).",
        "age_label": "6-36 months", "eligible": True,
        "energy_density": (None, None, "kcal/serve", "<=50kcal per serve -- a different basis to the kcal/100g used by most other categories; WHO NPPM does not define a fixed serve size in grams, so this cannot be converted to kcal/100g."),
        "sodium": ("lte", 50, "mg/100kcal", None),
        "total_sugar_pct_energy": (None, None, "%energy", "No separate %energy sugar cap specified; added free sugar itself is banned outright (see added_sugar)."),
        "added_sugar": ("not_permitted", None, None, "No added free sugars or sweeteners permitted at all."),
        "total_protein": (None, None, "g/100kcal", "No protein requirement for this category."),
        "total_fat": ("lte", 4.5, "g/100kcal", None),
        "fruit_content_pct_weight": ("eq", 100, "%weight", "Dry-fruit-only variant of this category (100% fruit) has this fruit-content requirement; fresh-fruit variant is fruit by definition."),
        "high_sugar_flag_note": "Flag required if sugar exceeds the 30% total-energy variant (=7.5g/100kcal, footnote 8).",
    },
    {
        "code": "5.2", "name": "Dry or semi-dry snacks and finger foods",
        "description": "Any grain, starch, pulse/lentil or root vegetable snack such as cracker, bread, biscuit, pastry, cake or pancake. Includes rusks, crackers and biscuits to be eaten dry or pulverised with liquid.",
        "age_label": "6-36 months", "eligible": True,
        "energy_density": (None, None, "kcal/serve", "<=50kcal per serve -- a different basis to the kcal/100g used by most other categories; WHO NPPM does not define a fixed serve size in grams, so this cannot be converted to kcal/100g."),
        "sodium": ("lte", 50, "mg/100kcal", None),
        "total_sugar_pct_energy": ("lte", 15, "%energy", None),
        "added_sugar": ("not_permitted", None, None, "No added free sugars or sweeteners permitted at all."),
        "total_protein": ("lte", 5.5, "g/100kcal", "This is a CAP, not a minimum -- only applies if the product is a biscuit/rusk made with added milk or another high-protein food (footnote 12ii). No protein requirement otherwise."),
        "total_fat": ("lte", 4.5, "g/100kcal", None),
        "fruit_content_pct_weight": (None, None, "%weight", "Not applicable to this category."),
        "high_sugar_flag_note": "Not applicable -- this category already has an explicit <=15% total-sugar-%energy cap.",
    },
    {
        "code": "6", "name": "Ingredients",
        "description": "Ingredients for cooking or adding to food in small quantities, e.g. olive oil, stock cubes.",
        "age_label": "6-36 months", "eligible": True,
        "energy_density": (None, None, "kcal/100g", "No requirement for this category."),
        "sodium": ("lte", 50, "mg/100kcal", None),
        "total_sugar_pct_energy": (None, None, "%energy", "No separate %energy sugar cap specified; added free sugar itself is banned outright (see added_sugar)."),
        "added_sugar": ("not_permitted", None, None, "No added free sugars or sweeteners permitted at all."),
        "total_protein": (None, None, "g/100kcal", "No requirement for this category."),
        "total_fat": (None, None, "g/100kcal", "No requirement for this category."),
        "fruit_content_pct_weight": ("not_permitted", None, "%weight", "No fruit content requirement/allowance specified (shown as None in the source)."),
        "high_sugar_flag_note": "Not applicable.",
    },
]

EXCLUDED_CATEGORIES = [
    {"code": "7", "name": "Confectionery",
     "description": "Chocolates, sweets, liquorice, marzipan, fruit chews, etc."},
    {"code": "8", "name": "Drinks",
     "description": "Fruit juice and other sweetened or flavoured drinks. Excludes 100% fruit/vegetable puree, breastmilk substitutes or unsweetened milk/milk alternatives."},
]

NUTRIENT_UNIT_FALLBACK = {
    "energy_density": "kcal/100g", "sodium": "mg/100kcal",
    "total_sugar_pct_energy": "%energy", "added_sugar": None,
    "total_protein": "g/100kcal", "total_fat": "g/100kcal",
    "fruit_content_pct_weight": "%weight",
}
COND_MAP = {"gte": ">=", "lte": "<=", "eq": "=", "not_permitted": "not_permitted", None: "not_specified"}

rows = []
for cat in CATEGORIES:
    for nutrient in ["energy_density", "sodium", "total_sugar_pct_energy", "added_sugar",
                      "total_protein", "total_fat", "fruit_content_pct_weight"]:
        cond, value, unit, note = cat[nutrient]
        rows.append(dict(
            category_code=cat["code"], category_name=cat["name"],
            category_description=cat["description"], age_label=cat["age_label"],
            eligible_for_promotion=1, nutrient=nutrient, condition=COND_MAP[cond],
            value=value, unit=unit or NUTRIENT_UNIT_FALLBACK[nutrient],
            value_note=note, high_sugar_flag_note=cat["high_sugar_flag_note"],
            source=SOURCE,
        ))

for cat in EXCLUDED_CATEGORIES:
    rows.append(dict(
        category_code=cat["code"], category_name=cat["name"],
        category_description=cat["description"], age_label=None,
        eligible_for_promotion=0, nutrient=None, condition=None, value=None, unit=None,
        value_note="WHO NPPM states this category is 'Not appropriate for promotion' to infants/"
                    "young children at all -- no composition thresholds are given because the "
                    "category itself should not be marketed to this age group.",
        high_sugar_flag_note=None, source=SOURCE,
    ))

print(f"Prepared {len(rows)} who_nppm_thresholds rows ({len(CATEGORIES)} eligible categories "
      f"x 7 nutrient rows each, + {len(EXCLUDED_CATEGORIES)} excluded-category rows).")

local_tmp_db = Path(tempfile.mkdtemp()) / "who_nppm_thresholds.db"
conn = sqlite3.connect(local_tmp_db)
cur = conn.cursor()
cur.execute("""
CREATE TABLE who_nppm_thresholds (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    category_code TEXT NOT NULL,
    category_name TEXT NOT NULL,
    category_description TEXT,
    age_label TEXT,
    eligible_for_promotion INTEGER NOT NULL,
    nutrient TEXT,
    condition TEXT,
    value REAL,
    unit TEXT,
    value_note TEXT,
    high_sugar_flag_note TEXT,
    source TEXT NOT NULL
)
""")
cols = list(rows[0].keys())
placeholders = ",".join(["?"] * len(cols))
cur.executemany(
    f"INSERT INTO who_nppm_thresholds ({','.join(cols)}) VALUES ({placeholders})",
    [tuple(r[c] for c in cols) for r in rows]
)
conn.commit()

n = cur.execute("SELECT count(*) FROM who_nppm_thresholds").fetchone()[0]
print(f"who_nppm_thresholds.db built: {n} rows.")
print()
print("Categories included:")
for code, name, elig in cur.execute(
        "SELECT DISTINCT category_code, category_name, eligible_for_promotion "
        "FROM who_nppm_thresholds ORDER BY eligible_for_promotion DESC, category_code"):
    print(f"  {code:<6}{name:<55}{'eligible' if elig else 'EXCLUDED (not appropriate for promotion)'}")
print()
print("Distinct (nutrient, unit) pairs -- confirms unit consistency per nutrient:")
for nutrient, unit, cnt in cur.execute(
        "SELECT nutrient, unit, count(*) FROM who_nppm_thresholds WHERE nutrient IS NOT NULL "
        "GROUP BY nutrient, unit ORDER BY nutrient"):
    print(f"  {nutrient:<28}{str(unit):<12}{cnt}")
conn.close()

import hashlib
local_hash = hashlib.sha256(local_tmp_db.read_bytes()).hexdigest()
shutil.copy(local_tmp_db, OUT_DB)
mount_hash = hashlib.sha256(Path(OUT_DB).read_bytes()).hexdigest()
print()
print("Copied to:", OUT_DB)
print("local build sha256:", local_hash)
print("mount copy sha256: ", mount_hash)
print("MATCH" if local_hash == mount_hash else "MISMATCH -- DO NOT TRUST THIS FILE")


Prepared 86 who_nppm_thresholds rows (12 eligible categories x 7 nutrient rows each, + 2 excluded-category rows).
who_nppm_thresholds.db built: 86 rows.

Categories included:
  1     Dry cereals and starches                               eligible
  2     Dairy foods, desserts and cereals                      eligible
  3.1   Fruit-containing product (purees/smoothies/fruit desserts)eligible
  3.2   Vegetable-only product                                 eligible
  4.1   Savoury meal WITHOUT protein or cheese named           eligible
  4.2   Savoury meal WITH CHEESE named, no other protein       eligible
  4.3   Savoury meal, protein source NOT named first           eligible
  4.4   Savoury meal, protein source named FIRST               eligible
  4.5   Protein source is ONLY named food                      eligible
  5.1   Fruit snacks                                           eligible
  5.2   Dry or semi-dry snacks and finger foods                eligible
  6     Ingredients           

## Schema

| column | notes |
|---|---|
| category_code | `1`, `2`, `3.1`, `3.2`, `4.1`-`4.5`, `5.1`, `5.2`, `6`, `7`, `8` -- WHO NPPM's own category numbering |
| category_name | short label, e.g. "Savoury meal, protein source named FIRST" |
| category_description | the full defining rule for the category, e.g. what makes a product "4.4" and not "4.3" |
| age_label | `6-36 months`, narrower for purees (`6-12 months only`) per category |
| eligible_for_promotion | `1` for the 12 categories NPPM gives thresholds for; `0` for Confectionery (7) and Drinks (8), which WHO states outright should not be promoted to this age group |
| nutrient | `energy_density` / `sodium` / `total_sugar_pct_energy` / `added_sugar` / `total_protein` / `total_fat` / `fruit_content_pct_weight` -- one row per (category x nutrient) |
| condition | `>=` / `<=` / `=` / `not_permitted` / `not_specified` |
| value | the numeric threshold; NULL where `condition` is `not_permitted` or `not_specified` |
| unit | see the units-audit section below -- one consistent unit per nutrient, with two named, source-driven exceptions |
| value_note | every conditional exception transcribed from the source: cheese-naming sodium exception, protein cap-vs-minimum distinction, dual weight-percentage protein requirements, etc. |
| high_sugar_flag_note | WHO's separate front-of-pack "high in sugar" warning-label threshold, where it applies (distinct from, and in some categories tighter than, the `total_sugar_pct_energy` column) |
| source | fixed citation string for the whole table |

## Units -- unified per nutrient, two named exceptions kept (not force-converted)

Every nutrient in this table uses exactly **one unit across all 12 eligible categories** --
confirmed by grouping the built table on `(nutrient, unit)` in the build cell's own output above:

- `sodium` -- always `mg/100kcal`
- `total_protein` -- always `g/100kcal`
- `total_fat` -- always `g/100kcal`
- `total_sugar_pct_energy` -- always `%energy`
- `fruit_content_pct_weight` -- always `%weight`
- `added_sugar` -- always a blanket `not_permitted` (no unit; WHO bans any added free sugar
  outright in every category, so there is no numeric threshold to have a unit)

**`energy_density` is the one nutrient that genuinely cannot be forced into a single unit**,
because WHO itself defines it on three different bases depending on the category, and none of
the three convert into each other without information this table does not have:

- `kcal/100g` for 9 of the 12 categories (the same basis as `products.db`'s own
  `energy_kcal_100g` column, so these 9 rows *are* directly comparable to a product)
- `kcal/serve` for the two dry-snack categories (5.1, 5.2) -- WHO caps these at 50 kcal per
  serve, not per 100 g, and the source does not define a fixed serve size in grams, so this
  cannot be converted to `kcal/100g` without guessing a serving size
- `% added water` for category 3.2 (vegetable-only purees) -- a completely different kind of
  measurement (a limit on dilution, not an energy value), not convertible to kcal at all

Each of these three rows carries its unit and a `value_note` explaining the difference, so a
consumer of this table can filter to the 9 directly-comparable rows, or handle the other 3
separately -- rather than the table silently pretending they are all the same basis.

## Known, confirmed-empty coverage

`saturated_fat`, `carbohydrate` and `dietary_fibre` have **no rows at all** in this table --
WHO NPPM's Table 1 does not cover them (checked directly against the transcribed source; not a
build omission). Epic 3's comparison table falls back to `reference_values.db` (itself missing
Saturated fat at every age) for those three, per the AC's requirement to still show all 8
nutrients with an honest "no reference available" state rather than dropping rows.